# 01_04_make_fits_summary

## 필요한 모듈

이 프로젝트를 위해서는 아래의 모듈이 필요하다. 

> numpy, pandas, matplotlib, astropy, version_information

### 모듈 설치

1. 콘솔 창에서 모듈을 설치할 때는 아래와 같은 형식으로 입력하면 된다.

>pip install module_name==version

>conda install module_name==version

2. 주피터 노트북(코랩 포함)에 설치 할 때는 아래의 셀을 실행해서 실행되지 않은 모듈을 설치할 수 있다. (pip 기준) 만약 아나콘다 환경을 사용한다면 7행을 콘다 설치 명령어에 맞게 수정하면 된다.

In [1]:
import importlib, sys, subprocess
packages = "numpy, pandas, matplotlib, scipy, astropy, photutils, ccdproc, version_information" # required modules
pkgs = packages.split(", ")
for pkg in pkgs :
    if not importlib.util.find_spec(pkg):
        print(f"**** module {pkg} is not installed")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
    else: 
        print(f"**** module {pkg} is installed")

**** module numpy is installed
**** module pandas is installed
**** module matplotlib is installed
**** module scipy is installed
**** module astropy is installed
**** module photutils is installed
**** module ccdproc is installed
**** module version_information is installed


### 모듈 버전 확인

아래 셀을 실행하면 이 노트북을 실행한 파이썬 및 관련 모듈의 버전을 확인할 수 있다.

In [13]:
%load_ext version_information
import time
now = time.strftime("%Y-%m-%d %H:%M:%S (%Z = GMT%z)")
print(f"This notebook was generated at {now} ")

vv = %version_information {packages}
for i, pkg in enumerate(vv.packages):
    print(f"{i} {pkg[0]:10s} {pkg[1]:s}")

The version_information extension is already loaded. To reload it, use:
  %reload_ext version_information
This notebook was generated at 2023-02-02 12:44:47 (KST = GMT+0900) 
0 Python     3.8.16 64bit [GCC 11.2.0]
1 IPython    8.8.0
2 OS         Linux 5.15.0 58 generic x86_64 with glibc2.17
3 numpy      1.22.3
4 pandas     1.5.2
5 matplotlib 3.6.3
6 scipy      1.8.1
7 astropy    5.2.1
8 photutils  1.6.0
9 ccdproc    2.4.0
10 version_information 1.0.4


### import modules

In [1]:
from glob import glob
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.stats import sigma_clip
from ccdproc import combine, ccd_process, CCDData

import ysfitsutilpy as yfu
import ysphotutilpy as ypu
import ysvisutilpy as yvu

import astro_utilities
import Python_utilities

plt.rcParams.update({'figure.max_open_warning': 0})

In [12]:
#%%
BASEDIR = astro_utilities.base_dir
BASEDIR = Path(astro_utilities.CCD_obs_raw_dir) 
ccd_dirs = ["QSI683ws_1bin", "QSI683ws_2bin", "QSI683ws_3bin", 
            "ST-8300M_1bin", "ST-8300M_2bin", 
            "STF-8300M_1bin", "STF-8300M_2bin", 
            "STL-11000M_1bin", "STL-11000M_2bin",
            "STX-16803_1bin", "STX-16803_2bin" ]

for ccd_dir in ccd_dirs[:1] :
    #BASEDIR = BASEDIR / ccd_dirs[0]

    #BASEDIRs = sorted(Python_utilities.getFullnameListOfsubDir(BASEDIR))
    BASEDIRs = sorted(Python_utilities.getFullnameListOfallsubDirs(BASEDIR/ccd_dir))
    #print ("BASEDIRs: {}".format(BASEDIRs))
    print ("len(BASEDIRs): {}".format(len(BASEDIRs)))


len(BASEDIRs): 802


In [14]:
#%%
ccd_fpath = Path(f"{BASEDIR/ccd_dir}.csv")
if ccd_fpath.exists():
    print(f"{BASEDIR/ccd_dir}.csv is already exist...")
else : 
    summary_all = pd.DataFrame()
    for fpath in BASEDIRs[:] :
        #BASEDIR = Path(BASEDIRs[0])
        fpath = Path(fpath)
        save_fpath2 = fpath/f"{fpath.parts[-1]}.csv"
        save_fpath = fpath/f"summary_{fpath.parts[-1]}.csv"
        print (f"Starting...\n{fpath.name}")
        if save_fpath2.exists():
            os.remove(str(save_fpath2))
            print (f"{str(save_fpath2)} is deleted...")
        
        if save_fpath.exists():
            print(f"{str(save_fpath)} is already exist...")
            summary = pd.read_csv(str(save_fpath))
        
        else : 
            summary = yfu.make_summary(fpath/"*.fit*",
                        output = save_fpath,
                        verbose = False
                        )
            print(f"{save_fpath} is created...")
        summary_all = pd.concat([summary_all, summary], axis = 0)
    summary_all.to_csv(f"{BASEDIR/ccd_dir}.csv")
    summary_all.reset_index(inplace=True)
    print(f"{BASEDIR/ccd_dir}.csv is created...")

Starting...
Cal
/mnt/Rdata/CCD_obs/CCD_obs_raw/QSI683ws_1bin/Cal/summary_Cal.csv is created...
Starting...
-_Bias_-_2016-10-05_-_-_QSI683ws_-_1bin
/mnt/Rdata/CCD_obs/CCD_obs_raw/QSI683ws_1bin/Cal/-_Bias_-_2016-10-05_-_-_QSI683ws_-_1bin/summary_-_Bias_-_2016-10-05_-_-_QSI683ws_-_1bin.csv is already exist...
Starting...
-_Bias_-_2016-10-13_-_-_QSI683ws_-_1bin
/mnt/Rdata/CCD_obs/CCD_obs_raw/QSI683ws_1bin/Cal/-_Bias_-_2016-10-13_-_-_QSI683ws_-_1bin/summary_-_Bias_-_2016-10-13_-_-_QSI683ws_-_1bin.csv is already exist...
Starting...
-_Bias_-_2016-10-19_-_-_QSI683ws_-_1bin
/mnt/Rdata/CCD_obs/CCD_obs_raw/QSI683ws_1bin/Cal/-_Bias_-_2016-10-19_-_-_QSI683ws_-_1bin/summary_-_Bias_-_2016-10-19_-_-_QSI683ws_-_1bin.csv is already exist...
Starting...
-_Bias_-_2016-12-28_-_-_QSI683ws_-_1bin
/mnt/Rdata/CCD_obs/CCD_obs_raw/QSI683ws_1bin/Cal/-_Bias_-_2016-12-28_-_-_QSI683ws_-_1bin/summary_-_Bias_-_2016-12-28_-_-_QSI683ws_-_1bin.csv is already exist...
Starting...
-_Bias_-_2018-07-06_-_-_QSI683ws_-_1bin
/